# From Chirality to the Standard Model
## Master verification notebook

[Paper](https://github.com/emad-ii/chirality-standard-model/blob/main/paper/paper.pdf) · [Repository](https://github.com/emad-ii/chirality-standard-model) · [Verification notes](https://github.com/emad-ii/chirality-standard-model/blob/main/verification/README.md)

Follow the finite mathematics behind the paper: the exceptional grading census,
two independent $E_6$ reconstructions, the intrinsic intermediate-group ordering
and subgroup kernels.
Run all cells in order. Python recomputes every recorded result once; the following
sections unpack those results and reconstruct the roots and subsystem orbits directly.
The last section can also load and run the pinned Lean proofs.

The manuscript supplies the structural and representation-theoretic proofs.
Exact computation discharges the finite searches; Lean checks the stated finite
propositions. The branching and its net chiral index are statements about the
classified representation.

## 1 · Open the mathematical sources

In a local checkout, setup finds the repository automatically. In Colab, a public
repository downloads automatically. While this repository is private, download
**Code → Download ZIP** from GitHub and upload it when prompted. Alternatively,
set `ARCHIVE_PATH` to a ZIP you have already uploaded. No GitHub token is needed.

Use the notebook from the same checkout or ZIP. Setup reports two separate identities:
the **repository commit and tree**, covering the paper, notebook and proofs, and the
**mathematical certificate checksum**, binding the recorded computational inputs.
A matching certificate alone does not identify the whole repository version.
For ZIPs, every archived file is checked against the recorded Git tree before the
mathematical run. These are consistency checks, not a publisher signature.

`REPO_REF` defaults to the current `main`. To reproduce a particular revision, paste
its full 40-character commit identifier; setup enforces that pin for either a clean
checkout or a ZIP. In a local Jupyter session, set the environment variable
`CHIRALITY_REPO_REF` before launch to pin without editing the tracked notebook.
Local edits are reported explicitly. An already-extracted ZIP can still run the
mathematics, but its source identity is labelled unverified until you provide the
original ZIP through `ARCHIVE_PATH`; that route is required for a pinned ZIP run.
There is no LaTeX or hosted-build dependency.

In [ ]:
from pathlib import Path, PurePosixPath
from hashlib import sha256
from html import escape
import json, os, re, shutil, stat, subprocess, sys, tempfile, zipfile
from IPython.display import HTML, display

REPOSITORY = "https://github.com/emad-ii/chirality-standard-model.git"
REPO_REF = "main" # @param {type:"string"}
ARCHIVE_PATH = "" # @param {type:"string"}
REPO_REF = os.environ.get("CHIRALITY_REPO_REF", REPO_REF)

# Never retain a previous verification verdict when setup is rerun.
full_audit_passed = False
lean_build_passed = False
verified_outputs = {}
EXPECTED_CERTIFICATE_SHA256 = "fe1d2cc8880e16b02d00f09b4f429e7438b45e65443a56608914843cf99b2bd1"

def is_repository(folder):
    return (folder / "verification/python/run_all.py").is_file() and (folder / "CubicAnomaly.lean").is_file()

SOURCE_ARCHIVE = None

def open_archive(filename):
    global SOURCE_ARCHIVE
    target = Path(tempfile.mkdtemp(prefix="chirality-")).resolve()
    with zipfile.ZipFile(filename) as archive:
        entries = archive.infolist()
        if len(entries) > 5000 or sum(e.file_size for e in entries) > 100_000_000:
            raise RuntimeError("Please upload the source ZIP, without dependency or build folders.")
        names = set()
        for entry in entries:
            name = PurePosixPath(entry.filename)
            if entry.filename in names or (not entry.is_dir() and name.as_posix() != entry.filename):
                raise RuntimeError("Duplicate or noncanonical path in ZIP.")
            names.add(entry.filename)
            if name.is_absolute() or ".." in name.parts or "\\" in entry.filename or stat.S_ISLNK(entry.external_attr >> 16):
                raise RuntimeError("Unsafe path in ZIP; use Code → Download ZIP on GitHub.")
            resolved = (target / entry.filename).resolve()
            if target != resolved and target not in resolved.parents:
                raise RuntimeError("ZIP entry escapes the extraction folder.")
        archive.extractall(target)
    candidates = [p.parent for p in target.rglob("CubicAnomaly.lean") if is_repository(p.parent)]
    if len(candidates) != 1:
        raise RuntimeError("The ZIP must contain exactly one complete repository.")
    SOURCE_ARCHIVE = Path(filename)
    return candidates[0]

REPO_ROOT = None
if ARCHIVE_PATH:
    REPO_ROOT = open_archive(Path(ARCHIVE_PATH).expanduser())
else:
    for folder in (Path.cwd(), *Path.cwd().parents):
        if is_repository(folder):
            REPO_ROOT = folder
            break
if REPO_ROOT is None:
    destination = Path(tempfile.mkdtemp(prefix="chirality-download-")) / "source"
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._/-]*", REPO_REF):
        raise ValueError("Use a branch, tag, or full commit identifier for REPO_REF.")
    commands = [
        ["git", "init", "--quiet", str(destination)],
        ["git", "-C", str(destination), "remote", "add", "origin", REPOSITORY],
        ["git", "-C", str(destination), "fetch", "--depth", "1", "--", "origin", REPO_REF],
        ["git", "-C", str(destination), "checkout", "--quiet", "--detach", "FETCH_HEAD"],
    ]
    for command in commands:
        fetched = subprocess.run(command, capture_output=True, text=True, timeout=120,
                                 env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
        if fetched.returncode != 0:
            break
    if fetched.returncode == 0 and is_repository(destination):
        REPO_ROOT = destination
    else:
        try:
            from google.colab import files
        except ImportError:
            raise RuntimeError("Open this notebook inside a checkout, or set ARCHIVE_PATH to the repository ZIP.") from None
        print("Private repository: upload the ZIP downloaded from GitHub (Code → Download ZIP).")
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise RuntimeError("Upload exactly one repository ZIP.")
        filename, contents = next(iter(uploaded.items()))
        upload_path = Path(tempfile.mkdtemp(prefix="chirality-upload-")) / "repository.zip"
        upload_path.write_bytes(contents)
        REPO_ROOT = open_archive(upload_path)

CERTIFICATE_DIR = REPO_ROOT / "verification/certificates"
certificate_file = CERTIFICATE_DIR / "e6_extrema_certificate.json"
if sha256(certificate_file.read_bytes()).hexdigest() != EXPECTED_CERTIFICATE_SHA256:
    raise RuntimeError("The computational certificate differs from the notebook pin. Open the notebook from this ZIP or checkout.")
PYTHON_DIR = REPO_ROOT / "verification/python"
# A setup rerun may point at a different ZIP. Reload all project modules, so
# reported source identity and the code used by later cells cannot diverge.
for name, module in list(sys.modules.items()):
    location = getattr(module, "__file__", None)
    if location and Path(location).parent.parts[-2:] in (("verification", "python"), ("formal", "scripts")):
        del sys.modules[name]
sys.path[:] = [item for item in sys.path if Path(item).parts[-2:] not in
               (("verification", "python"), ("formal", "scripts"))]
sys.path.insert(0, str(PYTHON_DIR))
from source_snapshot import archive_revision, checkout_revision, extracted_revision, enforce_revision
if SOURCE_ARCHIVE is not None:
    source_identity = archive_revision(SOURCE_ARCHIVE)
elif (REPO_ROOT / ".git").exists():
    source_identity = checkout_revision(REPO_ROOT)
else:
    source_identity = extracted_revision(REPO_ROOT)
enforce_revision(source_identity, REPO_REF)
import verify_e6_extrema as e6
from verification_checks import require
from run_all import OUTPUT_CHECKS

print("Repository:", REPO_ROOT)
print("Commit:", source_identity["commit"])
print("Source tree:", source_identity["tree"], "—", source_identity["state"])
print("Certificate:", EXPECTED_CERTIFICATE_SHA256)
print("Setup complete. The next cell recomputes every recorded mathematical result.")

## 2 · Recompute the exact evidence

This runs 12 mathematical programs, regenerates both JSON certificates, checks
the certificate digest, cross-checks its arithmetic, and compares the complete
generated Lean payload byte-for-byte. Adversarial tests check that corrupted data
and stale or altered source archives are rejected, even without running Lean.
All calculations use exact integers or rational numbers. Expect a few minutes;
only two verification processes run at once.

The remaining sections show outputs from this successful run, rather than
repeating the expensive searches. A failed comparison stops the notebook.

In [ ]:
completed = subprocess.run(
    [sys.executable, str(PYTHON_DIR / "run_all.py"), "--jobs", "2"],
    cwd=REPO_ROOT, text=True, capture_output=True, timeout=3600,
)
print(completed.stdout)
if completed.returncode != 0:
    raise RuntimeError(completed.stderr or "Mathematical verification failed.")
require("PASS  all 16 mathematical checks" in completed.stdout)
verified_outputs = {
    script: (REPO_ROOT / "verification/expected" / fixture).read_text()
    for _, script, fixture in OUTPUT_CHECKS
}
full_audit_passed = True

def show_result(script):
    # These records were just matched byte-for-byte to independently executed programs.
    print(verified_outputs[script].strip())

## 3 · Reconstruct the $E_6$ roots

Start with the six Bourbaki simple roots and repeatedly apply the simple
reflections defined by the Cartan matrix. The closure has 72 roots; 36 are
positive, and the highest root has coefficients $(1,2,2,3,2,1)$.

In [ ]:
roots = e6.orbit(e6.E[0], e6.reflect_root)
positive_roots = [root for root in roots if min(root) >= 0]
highest_root = max(positive_roots, key=sum)

require(len(roots) == 72)
require(len(positive_roots) == 36)
require(highest_root == (1, 2, 2, 3, 2, 1))
require(all(e6.root_inner(root, root) == 2 for root in roots))

root_summary = {
    "roots": len(roots),
    "positive roots": len(positive_roots),
    "highest-root coefficients": highest_root,
    "squared lengths": sorted({e6.root_inner(root, root) for root in roots}),
}
root_summary

## 4 · Build the subsystem universe

The minuscule-weight stabilizers give 27 copies of $D_5$. Root-line stabilizers
give 36 copies of $A_5+A_1$. The highest-root coefficient modulo three seeds
40 copies of $A_2^3$. Their Cartesian product contains $27\cdot36\cdot40=38{,}880$
triples. The separate complete closed-subsystem censuses verify that these are
all the maximal proper closed subsystem types.

The two complete censuses use different algorithms: positive-simple-root
enumeration in Cartan coordinates, and repeated root-pair adjoining with additive
closure in eight-dimensional coordinates. Both recover 5,079 closed subsystems;
the latter derives exact component ranks and Dynkin graphs for every one. Both
incidence implementations also traverse all six Weyl orbits, not just the extrema.

In [ ]:
weights = e6.orbit(e6.E[0], e6.reflect_weight)
d_by_weight = {
    weight: frozenset(root for root in roots if e6.pairing(weight, root) == 0)
    for weight in weights
}
root_pairs = {min(root, e6.neg(root)) for root in roots}
a_by_root = {
    root: frozenset(
        other
        for other in roots
        if other in (root, e6.neg(root)) or e6.root_inner(root, other) == 0
    )
    for root in root_pairs
}
theta_seed = frozenset(root for root in roots if root[highest_root.index(3)] % 3 == 0)
theta_orbit = e6.orbit(theta_seed, e6.reflect_set)

d_orbit = frozenset(d_by_weight.values())
a_orbit = frozenset(a_by_root.values())

require((len(d_orbit), len(a_orbit), len(theta_orbit)) == (27, 36, 40))
require({len(item) for item in d_orbit} == {40})
require({len(item) for item in a_orbit} == {32})
require({len(item) for item in theta_orbit} == {18})

triple_space = len(d_orbit) * len(a_orbit) * len(theta_orbit)
require(triple_space == 38_880)

orbit_summary = {
    "D5 orbit": len(d_orbit),
    "A5+A1 orbit": len(a_orbit),
    "A2^3 orbit": len(theta_orbit),
    "complete triple space": triple_space,
}
orbit_summary

## 5 · Exhaust the exceptional gradings

The toral census enumerates root-lattice functionals modulo two and three.
The cubic selector evaluates the grade-one trace cubic on each semisimple
factor. Only the $E_6$ component in the $E_8\supset E_6+A_2$ row is cubic-null.
The paper's invariant-theory argument identifies the corresponding radical.

In [ ]:
show_result("check_toral_gradings.py")
show_result("check_cubic_radical_selection.py")

## 6 · Classify the incidence space twice

The Cartan-matrix search exhausts all triples, computes their intersections and
constructs six simultaneous Weyl orbits. The independent realization embeds
the roots in $\mathbb R^8$ and repeats the calculation in different coordinates.
Both routes were executed and matched to their complete recorded outputs above.

In [ ]:
show_result("verify_e6_extrema.py")
show_result("verify_e6_extrema_r8_fast.py")

## 7 · Read the certificate

The JSON contains the root coordinates, orbit witnesses and numerical summaries.
It was independently regenerated byte-for-byte by the earlier run. Lean validates
the witness data and derives its authored incidence claims; it does not accept
the JSON's claimed counts as proofs.

In [ ]:
certificate_path = CERTIFICATE_DIR / "e6_extrema_certificate.json"
certificate_bytes = certificate_path.read_bytes()
certificate = json.loads(certificate_bytes)
recorded_digest = (CERTIFICATE_DIR / "e6_extrema_certificate.sha256").read_text().split()[0]
actual_digest = sha256(certificate_bytes).hexdigest()

require(certificate["certificate_version"] == 8)
require(recorded_digest == actual_digest)

finite = certificate["finite_object_counts"]
extrema = certificate["extrema"]
summary = {
    "roots": finite["roots"],
    "triple space": finite["triple_space"],
    "Weyl cells": len(certificate["joint_distribution"]),
    "pairwise maximizers": extrema["pairwise"]["maximizers"],
    "common maximizers": extrema["common"]["maximizers"],
    "labelled containment incidences": extrema["nested_pairs"]["labelled_ordered_pairs"],
}
expected_summary = {
    "roots": 72,
    "triple space": 38_880,
    "Weyl cells": 6,
    "pairwise maximizers": 4_320,
    "common maximizers": 2_160,
    "labelled containment incidences": 8_640,
}
require(summary == expected_summary)

rows = "".join(
    f"<tr><td style='padding:6px 18px 6px 0'>{escape(label)}</td>"
    f"<td style='padding:6px 0;text-align:right'><b>{value:,}</b></td></tr>"
    for label, value in summary.items()
)
display(HTML(f"""
<div style="display:inline-block;border:1px solid #d7c28d;border-radius:12px;
            padding:16px 22px;background:#fffaf0;min-width:360px">
  <div style="color:#8a5b13;font-size:12px;font-weight:700;letter-spacing:.08em">
    CERTIFICATE CHECKSUM CONSISTENT
  </div>
  <table style="margin-top:8px">{rows}</table>
  <div style="margin-top:10px;color:#566;font-size:12px">
    SHA-256 <code>{actual_digest[:16]}…{actual_digest[-8:]}</code>
  </div>
</div>
"""))

## 8 · Select the intermediate group

For each triple, form the three pairwise root intersections. Their derived
semisimple groups have dimension **rank + number of roots**. Calculate both
terms directly, then choose the unique group of greatest dimension.

The $D$–$A$ pair wins on every triple, including the nonextremal ones.
No family size enters this selection. The reproduced character calculation
then gives $d_\chi=15$ or $16$ for that group.

The two extremal orbits and this pair-first ordering specify the construction;
the paper proves its global subgroup forms and matter branchings.

In [ ]:
from collections import Counter
from functools import lru_cache
from itertools import product

@lru_cache(maxsize=None)
def semisimple_dimension(root_set):
    return e6.rank(root_set) + len(root_set)

ordering_cells = Counter()
for d_roots, a_roots, theta_roots in product(d_orbit, a_orbit, theta_orbit):
    pairs = (d_roots & a_roots, d_roots & theta_roots, a_roots & theta_roots)
    root_counts = tuple(map(len, pairs))
    dimensions = tuple(map(semisimple_dimension, pairs))
    require(root_counts[0] > max(root_counts[1:]), "Root-count ordering failed")
    require(dimensions[0] > max(dimensions[1:]), "Derived-group dimension ordering failed")
    common = len(d_roots & a_roots & theta_roots)
    ordering_cells[(root_counts, common, dimensions)] += 1

require(sum(ordering_cells.values()) == triple_space == 38_880)
require(len(ordering_cells) == 6)
headings = ("Pairwise roots", "Common roots", "dim H_DA", "dim H_DTheta", "dim H_ATheta", "Triples")
header = "".join(f"<th style='padding:8px'>{escape(h)}</th>" for h in headings)
rows = ""
for (profile, common, dimensions), count in sorted(ordering_cells.items()):
    values = (profile, common, *dimensions, f"{count:,}")
    rows += "<tr>" + "".join(f"<td style='padding:8px'>{escape(str(v))}</td>" for v in values) + "</tr>"
display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{rows}</tbody></table>"))
print("PASS: H_DA is uniquely largest on all 38,880 triples; no ties.")
show_result("check_e6_pair_orderings.py")

## 9 · Check the remaining steps

The closed-subsystem censuses support completeness. Context symmetries,
stabilizers and central kernels connect the finite geometry to the paper's
faithful subgroup construction. The rank-deficient checks implement
the finite exclusions and arithmetic accompanying the manuscript proofs.

Expand any result below to inspect its full reproduced output.

In [ ]:
already_shown = {"verify_e6_extrema.py", "verify_e6_extrema_r8_fast.py",
                 "check_toral_gradings.py", "check_cubic_radical_selection.py",
                 "check_e6_pair_orderings.py"}
for label, script, _ in OUTPUT_CHECKS:
    if script not in already_shown:
        display(HTML(f"<details><summary><b>{escape(label)}</b> — reproduced</summary>"
                     f"<pre>{escape(verified_outputs[script])}</pre></details>"))

## 10 · Check the finite proofs in Lean (optional)

Enable **RUN_LEAN** below. On a fresh Linux x86-64 Colab runtime, also enable
**INSTALL_LEAN** to install the checksum-verified Elan toolchain manager.
This step downloads Lean and Mathlib and needs additional time and disk space.
**USE_LEAN_CACHE** downloads upstream compiled dependencies when needed; turn
it off if you prefer to build the pinned sources.

The 25 exported theorems cover the exceptional Cartan census, cubic selector,
finite $E_6$ incidence calculation and intrinsic pair ordering. Pair ranks
are computed from root coordinates. The paper identifies rank plus root count
with Lie-algebra dimension. The `native_decide` axiom receipt is
`propext`, `Quot.sound`, `Lean.ofReduceBool`, `Lean.trustCompiler`.
The manuscript proves the compact-Lie and invariant-theoretic bridges.

In [ ]:
# @title Optional Lean proofs
RUN_LEAN = False # @param {type:"boolean"}
INSTALL_LEAN = False # @param {type:"boolean"}
USE_LEAN_CACHE = True # @param {type:"boolean"}

lean_build_passed = False
if RUN_LEAN:
    import platform, tarfile, urllib.request
    elan_bin = Path.home() / ".elan/bin"
    os.environ["PATH"] = str(elan_bin) + os.pathsep + os.environ.get("PATH", "")
    if shutil.which("lake") is None:
        if not INSTALL_LEAN:
            raise RuntimeError("Install Lean/Elan first, or enable INSTALL_LEAN on a Linux x86-64 Colab runtime.")
        if platform.system() != "Linux" or platform.machine() not in ("x86_64", "amd64"):
            raise RuntimeError("Automatic installation is for Linux x86-64. See https://lean-lang.org/install/ for your system.")
        installer_url = "https://github.com/leanprover/elan/releases/download/v4.2.4/elan-x86_64-unknown-linux-gnu.tar.gz"
        installer_sha = "42b94d4244e8353142c456ec0e4ca6528fd898a6c604d4059f494e706e431f63"
        folder = Path(tempfile.mkdtemp(prefix="chirality-elan-"))
        archive_path = folder / "elan.tar.gz"
        urllib.request.urlretrieve(installer_url, archive_path)
        require(sha256(archive_path.read_bytes()).hexdigest() == installer_sha, "Installer checksum mismatch")
        with tarfile.open(archive_path, "r:gz") as archive:
            members = [m for m in archive.getmembers() if m.isfile() and PurePosixPath(m.name).name == "elan-init"]
            require(len(members) == 1, "Missing or ambiguous Elan installer")
            installer = folder / "elan-init"
            installer.write_bytes(archive.extractfile(members[0]).read())
        installer.chmod(0o700)
        subprocess.run([str(installer), "-y", "--default-toolchain", "none"], check=True)
    lake = shutil.which("lake")
    require(lake is not None, "Lean installation did not provide lake")
    commands = [[sys.executable, "formal/scripts/check_generated_data.py"]]
    if USE_LEAN_CACHE and not (REPO_ROOT / ".lake/build/lib/lean/CubicAnomaly.olean").exists():
        commands.append([lake, "exe", "cache", "get"])
    commands.extend([[lake, "build"], [sys.executable, "formal/scripts/check_formal_hygiene.py"]])
    print("Building the pinned Lean/Mathlib project. A first run may take several minutes.")
    for command in commands:
        result = subprocess.run(command, cwd=REPO_ROOT, text=True, capture_output=True,
                                timeout=3600, env={**os.environ, "LEAN_NUM_THREADS": "2"})
        print(result.stdout.strip())
        if result.returncode != 0:
            raise RuntimeError(result.stderr or f"Command failed: {command}")
    lean_build_passed = True
    print("All 25 exported Lean theorems built; data and axiom checks passed.")
else:
    print("Lean not run. Enable RUN_LEAN above to check the formal proofs in this runtime.")

## Result

In [ ]:
require(full_audit_passed and summary == expected_summary)
require(root_summary["roots"] == 72 and orbit_summary["complete triple space"] == 38_880)
print("PASS: all 16 exact mathematical checks; 12 outputs and both certificates reproduced.")
print("PASS: 72 roots, 38,880 triples, six Weyl cells, 4,320/2,160 extrema, 8,640 labelled incidences.")
print("Lean:", "PASS — all 25 exported theorems and the axiom audit" if lean_build_passed else "not run in this session (optional)")
print("The paper supplies the structural, invariant-theoretic and representation-theoretic proofs.")